# Objetivo da exploração

Esta etapa tem como objetivo compreender a estrutura do dataset, identificar inconsistências, avaliar a qualidade dos dados e definir quais transformações serão necessárias antes da construção do dashboard no Power BI.

## Tecnologias
- Python
- Pandas
- Power BI

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/amazon.csv")

df.head()

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   product_id           1465 non-null   str  
 1   product_name         1465 non-null   str  
 2   category             1465 non-null   str  
 3   discounted_price     1465 non-null   str  
 4   actual_price         1465 non-null   str  
 5   discount_percentage  1465 non-null   str  
 6   rating               1465 non-null   str  
 7   rating_count         1463 non-null   str  
 8   about_product        1465 non-null   str  
 9   user_id              1465 non-null   str  
 10  user_name            1465 non-null   str  
 11  review_id            1465 non-null   str  
 12  review_title         1465 non-null   str  
 13  review_content       1465 non-null   str  
 14  img_link             1465 non-null   str  
 15  product_link         1465 non-null   str  
dtypes: str(16)
memory usage: 183.3 KB


## Análise Inicial

- O dataset possui 1.465 registros distribuídos em 16 colunas.
- A coluna `product_id` possui 1.351 identificadores únicos.
- A coluna `product_name` possui 1.337 nomes de produtos distintos.
- Foram identificadas repetições tanto em `product_id` quanto em `product_name`, indicando que alguns produtos aparecem em mais de um registro.
- O `product_id` mais frequente aparece 3 vezes.
- O `product_name` mais frequente aparece 5 vezes.
- A diferença entre a quantidade de valores únicos em `product_id` e `product_name` sugere que alguns produtos compartilham o mesmo nome comercial, mas possuem identificadores distintos. Uma hipótese é que representem variações de um mesmo produto, como cor, versão, fornecedor ou embalagem. Essa hipótese será investigada nas próximas etapas.


### Estrutura da coluna `category`

Foi observado que a coluna `category` não representa apenas uma categoria única, mas uma hierarquia de categorias separadas pelo caractere `|`.

Exemplo:

Electronics
    - WearableTechnology
        - SmartWatches

## Questões levantadas durante a exploração

- [x] Há valores ausentes no dataset?
- Não foram encontrados valores nulos.

- [x] Existem colunas com tipos incorretos?
- Sim. Colunas numéricas foram importadas como `str`.

- [X] Por que existem mais IDs únicos do que nomes únicos?
- O ID representa uma unidade individual de registro, o nome representa uma descrição que pode ser repetida

- [X] Existem produtos duplicados ou apenas variações do mesmo item?
-Apenas variações do mesmo item

- [x] Quais colunas precisarão de tratamento?
- discounted_price
- actual_price
- discount_percentage
- rating
- rating_count

### Problemas identificados

- Todas as colunas foram importadas como `str`.
- As colunas monetárias contêm o símbolo `₹`.
- A coluna `discount_percentage` contém o símbolo `%`.
- A coluna `rating_count` utiliza vírgulas como separadores de milhar.
- A coluna `rating` contém um valor inválido (`|`).
- Não foram identificados valores nulos durante a inspeção inicial. Entretanto, durante a etapa de limpeza foram encontrados dois registros com rating_count ausente, evidenciando que nem todas as inconsistências são detectadas na exploração preliminar.

### Inconsistência encontrada na coluna `rating`

Foi identificado um único valor inválido (`|`) na coluna `rating`. Como esse valor não representa uma avaliação numérica, ele será tratado antes da conversão da coluna para `float`.

Após inspeção da linha correspondente, verificou-se que as demais colunas apresentam valores consistentes. Dessa forma, apenas o campo `rating` necessitará de tratamento.

# Limpeza dos Dados

Nesta etapa serão realizadas as transformações necessárias para preparar o dataset para análise no Power BI.

Todas as modificações serão realizadas em uma cópia do DataFrame original, preservando os dados importados.

In [3]:
df_clean = df.copy()

## Tratamento da coluna `discounted_price`

A coluna `discounted_price` representa valores monetários, porém foi importada como `str` devido à presença do símbolo `₹` e do separador de milhar (`,`).

Esses caracteres serão removidos para permitir a conversão da coluna para o tipo `float`.

In [28]:
df_clean["discounted_price"].head()

0    399.0
1    199.0
2    199.0
3    329.0
4    154.0
Name: discounted_price, dtype: float64

In [27]:
df_clean["discounted_price"] = (
    df_clean["discounted_price"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

## Tratamento da coluna `actual_price`

A coluna `actual_price` representa o preço original dos produtos, porém foi importada como `str` devido à presença do símbolo `₹` e do separador de milhar (`,`).

Esses caracteres serão removidos para permitir a conversão da coluna para o tipo `float`.

In [11]:
df_clean["actual_price"].head()


0    1099.0
1     349.0
2    1899.0
3     699.0
4     399.0
Name: actual_price, dtype: float64

In [9]:
df_clean["actual_price"] = (
    df_clean["actual_price"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

## Tratamento da coluna `discount_percentage`

A coluna `discount_percentage` representa o percentual de desconto aplicado aos produtos.

Os valores foram importados como `str` devido à presença do símbolo `%`. Esse caractere será removido para permitir a conversão da coluna para o tipo `float`.

In [8]:
df_clean["discount_percentage"].describe()

count    1465.000000
mean       47.691468
std        21.635905
min         0.000000
25%        32.000000
50%        50.000000
75%        63.000000
max        94.000000
Name: discount_percentage, dtype: float64

In [7]:
df_clean["discount_percentage"] = (
    df_clean["discount_percentage"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

## Tratamento da coluna `rating`

A coluna `rating` representa a avaliação média dos produtos.

Durante a exploração dos dados foi identificado um único valor inválido (`|`), que não representa uma avaliação numérica. Esse valor será substituído por um valor ausente (`NaN`) antes da conversão da coluna para o tipo `float`.

In [6]:
df_clean["rating"].isna().sum()

np.int64(1)

In [5]:
df_clean["rating"] = df_clean["rating"].replace("|", pd.NA)
df_clean["rating"] = df_clean["rating"].astype(float)

## Tratamento da coluna `rating_count`

A coluna `rating_count` representa a quantidade de avaliações recebidas por cada produto.

Os valores foram importados como `str` devido à utilização de vírgulas como separadores de milhar. Essas vírgulas serão removidas para permitir a conversão da coluna para o tipo `int`.

Durante a etapa de limpeza foram identificados dois registros com valores ausentes na coluna `rating_count`.

Optou-se por manter esses valores como ausentes (`NaN`), em vez de substituí-los por zero, uma vez que não há evidências de que esses produtos realmente não possuam avaliações.

Essa abordagem preserva a integridade dos dados e evita introduzir informações não presentes no dataset original.

In [24]:
df_clean["rating_count"].head()

0    24269
1    43994
2     7928
3    94363
4    16905
Name: rating_count, dtype: Int64

In [20]:
df_clean["rating_count"] = (
    df_clean["rating_count"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

AttributeError: Can only use .str accessor with string values, not integer

In [17]:
mask = pd.to_numeric(
    df_clean["rating_count"].str.replace(",", "", regex=False),
    errors="coerce"
).isna()

df_clean.loc[mask, ["product_name", "rating_count"]]

,product_name,rating_count
282,Amazon Brand - Solimo 65W Fast Charging Braide...,NaN
324,"REDTECH USB-C to Lightning Cable 3.3FT, [Apple...",NaN


In [18]:
df_clean.loc[[282, 324]]

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
282,B0B94JPY2N,Amazon Brand - Solimo 65W Fast Charging Braide...,Computers&Accessories|Accessories&Peripherals|...,₹199,999.0,80.0,3.0,NaN,USB C to C Cable: This cable has type C connec...,AE7CFHY23VAJT2FI4NZKKP6GS2UQ,Pranav,RUB7U91HVZ30,The cable works but is not 65W as advertised,I have a pd supported car charger and I bought...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Amazon-Brand-Charging-Su...
324,B0BQRJ3C47,"REDTECH USB-C to Lightning Cable 3.3FT, [Apple...",Computers&Accessories|Accessories&Peripherals|...,₹249,999.0,75.0,5.0,NaN,💎[The Fastest Charge] - This iPhone USB C cabl...,AGJC5O5H5BBXWUV7WRIEIOOR3TVQ,Abdul Gafur,RQXD5SAMMPC6L,Awesome Product,Quick delivery.Awesome ProductPacking was good...,https://m.media-amazon.com/images/I/31-q0xhaTA...,https://www.amazon.in/REDTECH-Lightning-Certif...


In [21]:
df_clean["rating_count"] = (
    df_clean["rating_count"]
    .str.replace(",", "", regex=False)
)

df_clean["rating_count"] = pd.to_numeric(
    df_clean["rating_count"],
    errors="coerce"
).astype("Int64")

AttributeError: Can only use .str accessor with string values, not integer

# Validação da Limpeza

Após a realização das transformações, foram executadas algumas verificações para garantir que os tipos de dados foram convertidos corretamente e que as inconsistências identificadas durante a exploração foram tratadas.

In [29]:
df_clean.dtypes

product_id                 str
product_name               str
category                   str
discounted_price       float64
actual_price           float64
discount_percentage    float64
rating                 float64
rating_count             Int64
about_product              str
user_id                    str
user_name                  str
review_id                  str
review_title               str
review_content             str
img_link                   str
product_link               str
dtype: object

In [30]:
df_clean.isna().sum()

product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 1
rating_count           2
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
dtype: int64

In [31]:
df_clean.describe()

,discounted_price,actual_price,discount_percentage,rating,rating_count
count,1465.000000,1465.000000,1465.000000,1464.000000,1463.0
mean,3125.310874,5444.990635,47.691468,4.096585,18295.541353
std,6944.304394,10874.826864,21.635905,0.291674,42753.864952
min,39.000000,39.000000,0.000000,2.000000,2.0
25%,325.000000,800.000000,32.000000,4.000000,1186.0
50%,799.000000,1650.000000,50.000000,4.100000,5179.0
75%,1999.000000,4295.000000,63.000000,4.300000,17336.5
max,77990.000000,139900.000000,94.000000,5.000000,426973.0


# Conclusão da Limpeza

A etapa de limpeza permitiu preparar o dataset para as próximas fases do projeto.

Durante o processo foram realizadas as seguintes transformações:

- Conversão das colunas monetárias para o tipo `float`.
- Conversão da coluna de percentual para `float`.
- Conversão da coluna de avaliações (`rating`) para `float`.
- Conversão da coluna `rating_count` para o tipo `Int64`, preservando valores ausentes.
- Remoção de caracteres especiais (`₹`, `%` e separadores de milhar).
- Tratamento do valor inválido identificado na coluna `rating`.
- Identificação e preservação de valores ausentes em `rating_count`.
- Validação dos tipos de dados e da consistência das informações.

Ao final desta etapa, o dataset encontra-se preparado para a construção do dashboard no Power BI.

In [32]:
df_clean.to_csv(
    "../data/processed/amazon_clean.csv",
    index=False
)